In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))

import inventory as I
import manifest as M
import assemble as A
import concepts as C

smh_dir = Path("/home/jormond/Documents/DS217/users/jormond/SMH")
rbh_dir = Path("/home/jormond/Documents/DS217/users/jormond/RBH")
gosh_dir = Path("/home/jormond/Documents/DS217/users/jormond/GOSH")

### What is in the assembled tables
Prints, for each site, every assembled table with its row count and column names (no values). RBH tables are mostly wide (one column per measurement) and GOSH has one file per variable, so these names are what the concept map will be written from.

In [ ]:
assembled_dir = open_picu_dir / "derived" / "assembled"
for site in ["RBH", "GOSH"]:
    C.list_sources(assembled_dir, site)

### Cross-site concept map
Lists the long GOSH label columns first (so GOSH blood gases, fluids and drugs can be mapped), then measures every source in the map at all three sites: stays with a value, rows, share numeric and the 5th, median and 95th percentile. The percentiles show unit differences between sites. Counts and ranges only. This reads every assembled table, so it takes a few minutes.

In [ ]:
C.list_values(assembled_dir, "GOSH", "merged_vent_data__vent_Ventilation_mode_flowsheet_rows",
              "res_value", where=("var_type", "Ventilation"))

In [ ]:
G = "GOSH"
C.list_values(assembled_dir, G, "merged_gas_data__merged_blood_gasses", "standard_variable")
C.list_values(assembled_dir, G, "merged_balance__Net_Bal", "shortLabel")
C.list_values(assembled_dir, G, "merged_balance__Total_fluids", "shortLabel")
C.list_values(assembled_dir, G, "merged_vent_data__vent_Ventilation_mode_flowsheet_rows", "DisplayName")

In [ ]:
pooled_static = pd.read_parquet(open_picu_dir / "derived" / "static" / "pooled_static.parquet")
detail, summary = C.build(assembled_dir, static=pooled_static,
                          save_to=open_picu_dir / "derived" / "concepts" / "concept_map.csv")
pd.set_option("display.max_rows", 100)
print(summary.drop(columns="note").to_string(index=False))

### Coverage by admission year
For a few concepts, the percentage of each site's stays with the concept in each admission year. A source that only exists in some years (for example a system change) shows as a step.

In [ ]:
chk = ["bg_ph", "vent_mode", "tidal_volume", "ie_ratio", "net_balance", "etco2", "abp_sys", "fio2"]
yr, piv = C.by_year(assembled_dir, pooled_static, chk)
print(piv.to_string())

In [ ]:
A.list_tables("GOSH", gosh_dir, include=r"raw_lab_values|lab_components|Archived", exclude="")

In [ ]:
import pyarrow.parquet as pq
pq.ParquetFile(gosh_dir / "gosh_mech_vent_31March2026" / "final_gosh_dataset.parquet").schema.names

In [ ]:
yr, piv = C.by_year(assembled_dir, pooled_static, list(C.CONCEPTS), sites=("GOSH",))

def share(a, b):
    d = yr[yr.year.between(a, b)].groupby("concept")[["stays", "n"]].sum()
    return (100 * d.stays / d.n).round(1)

era = pd.DataFrame({"pre_epic_pct": share(2013, 2018), "post_epic_pct": share(2020, 2022)})
print(era[(era.pre_epic_pct >= 50) & (era.post_epic_pct < 0.5 * era.pre_epic_pct)].to_string())

In [ ]:
cols = ["hour_timestamp", "pH", "ventMode", "TV_measured", "IERatio"]
old = pd.read_parquet(gosh_dir / "gosh_mech_vent_31March2026" / "final_gosh_dataset.parquet", columns=cols)
old["year"] = pd.to_datetime(old["hour_timestamp"], errors="coerce").dt.year
print(old.groupby("year")[["pH", "ventMode", "TV_measured", "IERatio"]].agg(lambda s: round(100 * s.notna().mean(), 1)).to_string())

### Does the old GOSH parquet hold post-2019 data we are missing?
`final_gosh_dataset.parquet` is an older hourly dataset built from the GOSH sources. For every column, this shows how many hourly rows have a value before and after the CareVue to Epic switch (about 20 April 2019). If blood gas, vent mode, tidal volume or I:E ratio have values after 2019 here, they exist in a source we have not assembled. Counts only.

In [ ]:
old_path = gosh_dir / "gosh_mech_vent_31March2026" / "final_gosh_dataset.parquet"
era = C.era_columns(old_path)
print(era.sort_values("post_pct").to_string())

In [ ]:
g = pooled_static[pooled_static.site == "GOSH"]
print((pd.to_datetime(g["t0"]) >= "2019-04-20").sum(), len(g))